# Aligned Education — LLM-Judge Prompt Optimiser

Iteratively aligns a judge prompt with human-expert golden labels from
`education_golden_test_cases.xlsx`.

**Flow**
1. Start with `prompt_0` (based on *System Prompt 1*).
2. For each round, send every test-case row to the webhook with `{"input": filled_prompt}`.
3. Parse the JSON response; compare predicted labels against golden labels.
4. If **all** fields have mismatch rate ≤ 30 % → stop.
5. Otherwise feed mismatched rows into `OptPrompt` to obtain an improved prompt, then repeat.
6. Stop after **5 rounds** at most.

In [1]:
import os
import re
import json
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
N8N_BASE_URL       = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH       = "9d6049fc-77c8-43e9-b71e-f734506f4f9d"   # ← fill in
USE_TEST_URL       = False   # True → /webhook-test/…  |  False → /webhook/…

GOLDEN_FILE        = "golden_test_cases/education_golden_test_cases.xlsx"
RESULT_DIR         = "education_test_result"

MAX_ROUNDS         = 5
MISMATCH_THRESHOLD = 0.30    # stop when ALL fields are within this rate
TIMEOUT            = 600      # seconds per request (10 minutes)
RETRIES            = 2
DELAY_BETWEEN      = 0.5     # seconds between rows

SCORE_FIELDS = ["faithfulness", "relevance", "simplicity", "retrieval"]


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/9d6049fc-77c8-43e9-b71e-f734506f4f9d


## OptPrompt

Meta-prompt sent to the webhook when mismatches exceed the threshold.
Placeholders: **`{oldPrompt}`**, **`{mismatchResults}`**.
The LLM must return an *UpdatePrompt* whose own placeholders are
`{userMessage}`, `{actualResult}`, `{retrievedText}`.

In [2]:
OptPrompt = (
    "You are an expert prompt engineer specialising in LLM-as-a-judge systems "
    "for evaluating Thai bank chatbot responses.\n\n"
    "Your task is to improve the judge prompt below so it better aligns with "
    "human-expert (golden) evaluations.\n\n"
    "## Current Prompt\n"
    "{oldPrompt}\n\n"
    "## Mismatch Cases\n"
    "The following rows show where the current prompt disagreed with golden labels. "
    "Each entry includes the original inputs, the golden label, and the predicted label:\n"
    "{mismatchResults}\n\n"
    "## Your Task\n"
    "1. Analyse the mismatch patterns.\n"
    "2. Identify what the current prompt misinterprets.\n"
    "3. Rewrite the prompt to fix those issues while preserving correct behaviours.\n\n"
    "## Requirements for the output prompt\n"
    "- MUST keep exactly these placeholders (with single curly braces): "
    "{userMessage}, {actualResult}, {retrievedText}\n"
    "- MUST instruct the LLM to return ONLY a JSON object with these fields and allowed values:\n"
    '    { "faithfulness": "good"|"acceptable"|"invalid",\n'
    '      "relevance":    "good"|"acceptable"|"invalid",\n'
    '      "simplicity":  "good"|"acceptable"|"invalid",\n'
    '      "retrieval":   "good"|"acceptable"|"invalid" }\n\n'
    "Return ONLY the improved prompt text — no preamble, no explanation."
)

print("OptPrompt defined:", len(OptPrompt), "chars")

OptPrompt defined: 1157 chars


## Initial Judge Prompt — `prompt_0`

Adapted from *System Prompt 1* in `raw_ingredient_evaluation/`.
Uses **`{userMessage}`**, **`{actualResult}`**, **`{retrievedText}`** as placeholders.
Returns JSON with `faithfulness`, `relevance`, `simplicity`, `retrieval`,
each valued **`"good"`** / **`"acceptable"`** / **`"invalid"`**.

In [3]:
prompt_0 = (
    "You are a Quality Assurance (QA) expert evaluating AI Chatbot responses for a Thai bank.\n"
    "Your evaluation follows \"Fact accuracy first\" and \"Safety over definitive rejection\"\n"
    "under the \"Acceptable at 3\" framework.\n\n"
    "## Evaluation Criteria\n\n"
    "### 1. faithfulness — accuracy against the retrieved context\n"
    "- \"good\"       : All information is 100 % correct per the retrieved context.\n"
    "- \"acceptable\" : Content is principally correct. Includes Safe Responses where the bot\n"
    "                   lists correct eligibility conditions for the customer to self-verify,\n"
    "                   even if the customer's data conflicts with some conditions.\n"
    "                   Do NOT treat this as hallucination — the bot is acting safely.\n"
    "- \"invalid\"    : Specific numbers/conditions contradict the context, or the bot invents\n"
    "                   sources/data that do not exist.\n\n"
    "### 2. relevance — how well the response addresses the question\n"
    "- \"good\"       : Directly answers the question, correctly analyses customer data, and\n"
    "                   reaches an accurate conclusion.\n"
    "- \"acceptable\" : The bot is uncertain or avoids a definitive rejection, instead listing\n"
    "                   correct key conditions for the customer to judge themselves.\n"
    "                   This reduces the risk of a wrongful rejection.\n"
    "- \"invalid\"    : Targets the wrong topic, or definitively confirms the customer passes\n"
    "                   eligibility when their data clearly contradicts the document.\n\n"
    "### 3. simplicity — readability and communication quality\n"
    "- \"good\"       : Natural language, concise, well structured.\n"
    "- \"acceptable\" : Easy to follow, even if lengthy or presented as a copied list.\n"
    "- \"invalid\"    : Repetitive, circular, or uses incomprehensible language.\n\n"
    "### 4. retrieval — quality of the context that was retrieved\n"
    "- \"good\"       : Retrieved context is directly relevant or from the correct programme.\n"
    "- \"acceptable\" : Context is on the right topic even if some noise is mixed in.\n"
    "- \"invalid\"    : Context is irrelevant or from the wrong programme.\n\n"
    "## Special Rule — Safety Over Rejection\n"
    "If the chatbot responds with a conditional list of correct requirements instead of a direct\n"
    "answer (Safe/Conditional Response), both \"faithfulness\" and \"relevance\" must be at least\n"
    "\"acceptable\", provided the listed conditions are correct according to the context.\n\n"
    "## Input Data\n\n"
    "Retrieved Context:\n"
    "{retrievedText}\n\n"
    "Customer Question:\n"
    "{userMessage}\n\n"
    "Chatbot Response:\n"
    "{actualResult}\n\n"
    "## Output Instructions\n"
    "Return ONLY a JSON object — no markdown, no explanation:\n"
    '{\n'
    '  "faithfulness": "good" | "acceptable" | "invalid",\n'
    '  "relevance":    "good" | "acceptable" | "invalid",\n'
    '  "simplicity":  "good" | "acceptable" | "invalid",\n'
    '  "retrieval":   "good" | "acceptable" | "invalid"\n'
    '}'
)

print("prompt_0 defined:", len(prompt_0), "chars")

prompt_0 defined: 2743 chars


## Helper Functions

In [4]:
def fill_prompt(template: str, userMessage: str, actualResult: str, retrievedText: str) -> str:
    return (template
            .replace("{userMessage}",   str(userMessage))
            .replace("{actualResult}",  str(actualResult))
            .replace("{retrievedText}", str(retrievedText)))


def fill_opt_prompt(template: str, oldPrompt: str, mismatchResults: str) -> str:
    return (template
            .replace("{oldPrompt}",       str(oldPrompt))
            .replace("{mismatchResults}", str(mismatchResults)))


def _call_raw(payload: dict, timeout: int = TIMEOUT, retries: int = RETRIES):
    url = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


def call_webhook(prompt: str):
    return _call_raw({"input": prompt})


def get_response_text(resp) -> str:
    if isinstance(resp, str):
        return resp
    if isinstance(resp, dict):
        for key in ("output", "text", "result", "response", "content", "message"):
            if key in resp and resp[key] is not None:
                return str(resp[key])
        return json.dumps(resp, ensure_ascii=False)
    return str(resp)


def parse_json_response(resp) -> dict:
    text = get_response_text(resp)
    # 1. markdown code block
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # 2. whole text as JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 3. first JSON object found
    m = re.search(r"(\{[\s\S]*?\})", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return {}


def extract_prompt_text(resp) -> str:
    text = get_response_text(resp)
    m = re.search(r"```(?:\w+)?\s*([\s\S]*?)\s*```", text)
    if m:
        return m.group(1).strip()
    return text.strip()


print("Helpers ready.")

Helpers ready.


## Load Golden Test Cases

In [5]:
df_golden = pd.read_excel(GOLDEN_FILE)
print(f"Loaded {len(df_golden)} golden test cases")
print("Columns:", df_golden.columns.tolist())
print("\nLabel distributions:")
for f in SCORE_FIELDS:
    print(f"  {f}: {df_golden[f].value_counts().to_dict()}")
df_golden.head(3)

Loaded 30 golden test cases
Columns: ['ragFocus', 'testId', 'userMessage', 'actualResult', 'retrievedText', 'overall_quality', 'faithfulness', 'faithfulness_reason', 'relevance', 'relevance_reason', 'simplicity', 'simplicity_reason', 'retrieval', 'retrieve_reason', 'improvement_suggestion']

Label distributions:
  faithfulness: {'good': 21, 'invalid': 8, 'acceptable': 1}
  relevance: {'acceptable': 18, 'good': 12}
  simplicity: {'good': 16, 'acceptable': 14}
  retrieval: {'good': 21, 'acceptable': 1}


,ragFocus,testId,userMessage,actualResult,retrievedText,overall_quality,faithfulness,faithfulness_reason,relevance,relevance_reason,simplicity,simplicity_reason,retrieval,retrieve_reason,improvement_suggestion
0,คุณสู้,TC-001,ผ่อนบ้านมา 2 ปี ยอดกู้ 3.5 ล้าน ตอนนี้ตกงานค้า...,คุณสามารถพิจารณาเข้าร่วมโครงการ “คุณสู้ เราช่ว...,[file_name: khunsoo-v3p0.txt | lines: 1-6]\nโค...,good,good,เนื้อหาเป็นไปตามเอกสารไม่ออกนอกลู่นอกทาง,acceptable,บอกชื่อโครงการถูก แต่ควรบอกเสริมว่ามีเงื่อนไขอ...,good,เขียนเข้าใจง่าย,good,Context ที่ดึงมา (khunsoo-v3p0.txt) เกี่ยวข้อง...,ควรเพิ่มรายละเอียดเกี่ยวกับรูปแบบการช่วยเหลือ ...
1,คุณสู้,TC-002,อยากเข้าโครงการที่ลดค่างวดครึ่งนึงค่ะ ผ่อนรถยน...,คุณไม่สามารถเข้าร่วมมาตรการที่ 1 “จ่ายตรง คงทร...,[file_name: khunsoo-v3p0.txt | lines: 1-6]\nโค...,good,good,ข้อมูลถูกต้อง,good,ตอบได้ครงคำถาม,good,เขียนเข้าใจง่าย,good,Context ที่ดึงมาเกี่ยวข้องโดยตรงกับมาตรการที่ ...,ไม่มี
2,คุณสู้,TC-003,พิ่งกู้ซื้อคอนโด 3 ล้าน เมื่อกุมภาปี 67 ตอนนี้...,คุณไม่สามารถเข้าร่วมมาตรการภายใต้โครงการ “คุณส...,[file_name: khunsoo-v3p0.txt | lines: 1-6]\nโค...,good,good,ตอบได้ตรงเอกสาร,good,ตอบได้ครงคำถาม,good,ตอบกระชับได้ใจความ,good,Context ที่ดึงมาเกี่ยวข้องโดยตรงกับคำถามและมาต...,ไม่มีคำแนะนำเพิ่มเติม เนื่องจากคำตอบถูกต้องและ...


## Optimisation Loop

For each round:
1. Fill `{userMessage}`, `{actualResult}`, `{retrievedText}` in the current prompt.
2. POST `{"input": filled_prompt}` to the webhook; collect responses.
3. Parse each response string → dict with `re` / `json`.
4. Save `test_result_prompt_N.xlsx`.
5. Compute per-field mismatch rate vs golden labels.
6. **Stop** if every field's rate ≤ 30 %, or after 5 rounds.
7. Otherwise collect mismatch rows → fill `OptPrompt` → POST to webhook → use response as next prompt.

In [6]:
os.makedirs(RESULT_DIR, exist_ok=True)

current_prompt = prompt_0
all_rounds: list[dict] = []

for round_idx in range(MAX_ROUNDS):
    round_name = f"prompt_{round_idx}"
    out_path   = os.path.join(RESULT_DIR, f"test_result_{round_name}.xlsx")
    prompt_path = os.path.join(RESULT_DIR, f"{round_name}.txt")

    print(f"\n{'=' * 60}")
    print(f"Round {round_idx}  |  {round_name}")
    print(f"{'=' * 60}")

    # ── Save prompt as text file ───────────────────────────────────────────
    with open(prompt_path, "w", encoding="utf-8") as fh:
        fh.write(current_prompt)
    print(f"  Saved prompt → {prompt_path}")

    # ── Check if result xlsx already exists — skip evaluation if so ───────
    if os.path.exists(out_path):
        print(f"  Result already exists → {out_path}  (skipping evaluation)")
        df_test_result = pd.read_excel(out_path)
        df_pred = pd.DataFrame({f: df_test_result[f"pred_{f}"].values for f in SCORE_FIELDS})
    else:
        # ── 1. Run evaluation ──────────────────────────────────────────────
        predicted: list[dict] = []
        errors:    list[str | None] = []

        for _, row in tqdm(df_golden.iterrows(), total=len(df_golden), desc=round_name):
            try:
                filled  = fill_prompt(current_prompt,
                                       row["userMessage"],
                                       row["actualResult"],
                                       row["retrievedText"])
                raw     = call_webhook(filled)
                parsed  = parse_json_response(raw)
                predicted.append({f: parsed.get(f) for f in SCORE_FIELDS})
                errors.append(None)
            except Exception as exc:
                predicted.append({f: None for f in SCORE_FIELDS})
                errors.append(str(exc))
            time.sleep(DELAY_BETWEEN)

        df_pred = pd.DataFrame(predicted)

        # ── 2. Build test_result dataframe ────────────────────────────────
        df_test_result = df_golden[["testId", "ragFocus", "userMessage"] + SCORE_FIELDS].copy()
        for f in SCORE_FIELDS:
            df_test_result[f"pred_{f}"] = df_pred[f].values
            df_test_result[f"match_{f}"] = (df_pred[f].values == df_golden[f].values)
        df_test_result["error"] = errors

        df_test_result.to_excel(out_path, index=False)
        print(f"  Saved → {out_path}")

    # ── 3. Compute per-field mismatch rates ───────────────────────────────
    mismatch_rates: dict[str, float] = {}
    for f in SCORE_FIELDS:
        valid = df_golden[f].notna() & df_pred[f].notna()
        if valid.sum():
            mismatch_rates[f] = float((~df_test_result[f"match_{f}"][valid]).mean())
        else:
            mismatch_rates[f] = float("nan")

    print("\n  Mismatch rates:")
    for f, rate in mismatch_rates.items():
        tag = "✓" if (not pd.isna(rate) and rate <= MISMATCH_THRESHOLD) else "✗"
        print(f"    {f:20s}: {rate:.1%}  {tag}")

    all_rounds.append({
        "round":          round_idx,
        "prompt_name":    round_name,
        "prompt":         current_prompt,
        "df_test_result": df_test_result.copy(),
        "mismatch_rates": mismatch_rates.copy(),
    })

    # ── 4. Stopping condition ─────────────────────────────────────────────
    valid_rates = [v for v in mismatch_rates.values() if not pd.isna(v)]
    max_mismatch = max(valid_rates) if valid_rates else float("nan")

    if not pd.isna(max_mismatch) and max_mismatch <= MISMATCH_THRESHOLD:
        print(f"\n  All fields within {MISMATCH_THRESHOLD:.0%} threshold. Stopping.")
        break

    if round_idx == MAX_ROUNDS - 1:
        print(f"\n  Reached maximum {MAX_ROUNDS} rounds. Stopping.")
        break

    # ── 5. Collect mismatched rows ─────────────────────────────────────────
    mismatch_mask = pd.Series(False, index=df_golden.index)
    for f in SCORE_FIELDS:
        mismatch_mask |= (df_pred[f].values != df_golden[f].values)

    df_mismatch = (
        df_golden[mismatch_mask][
            ["testId", "userMessage", "actualResult", "retrievedText"] + SCORE_FIELDS
        ]
        .copy()
        .reset_index(drop=True)
    )
    df_pred_mis = df_pred[mismatch_mask.values].reset_index(drop=True)
    for f in SCORE_FIELDS:
        df_mismatch[f"pred_{f}"] = df_pred_mis[f].values

    mismatch_json = df_mismatch.to_json(orient="records", force_ascii=False, indent=2)
    print(f"\n  {mismatch_mask.sum()} mismatched rows → OptPrompt.")

    # ── 6. Get improved prompt ─────────────────────────────────────────────
    opt_filled = fill_opt_prompt(OptPrompt, current_prompt, mismatch_json)
    print("  Requesting improved prompt…")
    opt_resp = call_webhook(opt_filled)
    current_prompt = extract_prompt_text(opt_resp)
    print(f"  New prompt ({len(current_prompt)} chars) ready for round {round_idx + 1}.")

print(f"\nOptimisation complete — {len(all_rounds)} round(s) executed.")


Round 0  |  prompt_0
  Saved prompt → education_test_result/prompt_0.txt
  Result already exists → education_test_result/test_result_prompt_0.xlsx  (skipping evaluation)

  Mismatch rates:
    faithfulness        : 6.7%  ✓
    relevance           : 60.0%  ✗
    simplicity          : 46.7%  ✗
    retrieval           : 13.6%  ✓

  20 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (4223 chars) ready for round 1.

Round 1  |  prompt_1
  Saved prompt → education_test_result/prompt_1.txt


prompt_1:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → education_test_result/test_result_prompt_1.xlsx

  Mismatch rates:
    faithfulness        : 6.7%  ✓
    relevance           : 20.0%  ✓
    simplicity          : 46.7%  ✗
    retrieval           : 13.6%  ✓

  19 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (4997 chars) ready for round 2.

Round 2  |  prompt_2
  Saved prompt → education_test_result/prompt_2.txt


prompt_2:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → education_test_result/test_result_prompt_2.xlsx

  Mismatch rates:
    faithfulness        : 6.7%  ✓
    relevance           : 23.3%  ✓
    simplicity          : 43.3%  ✗
    retrieval           : 9.1%  ✓

  19 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (5649 chars) ready for round 3.

Round 3  |  prompt_3
  Saved prompt → education_test_result/prompt_3.txt


prompt_3:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → education_test_result/test_result_prompt_3.xlsx

  Mismatch rates:
    faithfulness        : 3.3%  ✓
    relevance           : 26.7%  ✓
    simplicity          : 46.7%  ✗
    retrieval           : 13.6%  ✓

  19 mismatched rows → OptPrompt.
  Requesting improved prompt…
  New prompt (6001 chars) ready for round 4.

Round 4  |  prompt_4
  Saved prompt → education_test_result/prompt_4.txt


prompt_4:   0%|          | 0/30 [00:00<?, ?it/s]

  Saved → education_test_result/test_result_prompt_4.xlsx

  Mismatch rates:
    faithfulness        : 10.0%  ✓
    relevance           : 23.3%  ✓
    simplicity          : 16.7%  ✓
    retrieval           : 9.1%  ✓

  All fields within 30% threshold. Stopping.

Optimisation complete — 5 round(s) executed.


## Results Summary

In [7]:
summary_rows = []
for r in all_rounds:
    row = {"round": r["round"], "prompt_name": r["prompt_name"]}
    row.update({f: f'{v:.1%}' for f, v in r["mismatch_rates"].items()})
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("=" * 60)
print("OPTIMISATION SUMMARY — mismatch rate per field")
print("=" * 60)
display(df_summary)

best = min(
    all_rounds,
    key=lambda r: max(
        (v for v in r["mismatch_rates"].values() if not pd.isna(v)),
        default=float("inf"),
    ),
)
print(f"\nBest round : {best['prompt_name']}")
print(f"Max mismatch: "
      f"{max(v for v in best['mismatch_rates'].values() if not pd.isna(v)):.1%}")
print("\nAccess results:")
print("  all_rounds[N]['df_test_result']  — labelled DataFrame for round N")
print("  all_rounds[N]['prompt']          — prompt text used in round N")
print("  all_rounds[-1]['prompt']         — final (best last) prompt")

OPTIMISATION SUMMARY — mismatch rate per field


,round,prompt_name,faithfulness,relevance,simplicity,retrieval
0,0,prompt_0,6.7%,60.0%,46.7%,13.6%
1,1,prompt_1,6.7%,20.0%,46.7%,13.6%
2,2,prompt_2,6.7%,23.3%,43.3%,9.1%
3,3,prompt_3,3.3%,26.7%,46.7%,13.6%
4,4,prompt_4,10.0%,23.3%,16.7%,9.1%



Best round : prompt_4
Max mismatch: 23.3%

Access results:
  all_rounds[N]['df_test_result']  — labelled DataFrame for round N
  all_rounds[N]['prompt']          — prompt text used in round N
  all_rounds[-1]['prompt']         — final (best last) prompt
